In [ ]:
import pandas as pd
adsl = pd.read_csv(r"C:\Users\AISWARYA\Desktop\ADNI\FROMR\ADSL.csv")
apoe = pd.read_csv(r"C:\Users\AISWARYA\Desktop\ADNI\FROMR\APOERES.csv")
adas = pd.read_csv(r"C:\Users\AISWARYA\Desktop\ADNI\FROMR\ADAS.csv")

In [ ]:
print("ADSL columns:\n", adsl.columns.tolist(), "\n")
print("APOERES columns:\n", apoe.columns.tolist(), "\n")
print("ADAS columns:\n", adas.columns.tolist(), "\n")

In [ ]:
#most important columns
for name, df in [("ADSL", adsl), ("APOERES", apoe), ("ADAS", adas)]:
    print(f"\n{name} preview:")
    cols_to_show = [c for c in ["PTID", "RID", "USUBJID", "SUBJID", "VISCODE", "AGE", "SEX", "EDUC", "APOE", "TOTSCORE", "TOTAL13"] if c in df.columns]
    print(df[cols_to_show].head())

In [ ]:
# ADSL: demographics
adsl_small = adsl[["USUBJID", "AGE", "SEX", "EDUC"]].copy()
# extract PTID from USUBJID 
adsl_small["PTID"] = adsl_small["USUBJID"].str.extract(r"(\d{3}_S_\d{4})")
# APOE: genetics
# adjust this if APOERES uses a slightly different APOE column name
apoe_cols = [c for c in ["RID", "PTID", "APOE", "APOE4"] if c in apoe.columns]
apoe_small = apoe[apoe_cols].copy()
# ADAS: cognition
adas_small = adas[["RID", "PTID", "VISCODE", "TOTSCORE", "TOTAL13"]].copy()

print(adsl_small.head())
print(apoe_small.head())
print(adas_small.head())

In [ ]:
dxsum = pd.read_csv(r"C:\Users\AISWARYA\Desktop\ADNI\DXSUM_26Jan2026.csv", low_memory=False)
print("DXSUM columns:\n", dxsum.columns.tolist(), "\n")
print(dxsum[["PTID", "RID", "VISCODE", "DIAGNOSIS"]].head())

In [ ]:
print("ADSL ID preview:")
print(adsl[["USUBJID", "SUBJID"]].head(10))
print("\nAPOERES ID preview:")
print(apoe[["PTID", "RID"]].head(10))

In [ ]:
adas_small = adas[["RID", "PTID", "VISCODE", "TOTSCORE", "TOTAL13"]].copy()
dx_small = dxsum[["RID", "PTID", "VISCODE", "DIAGNOSIS"]].copy()
apoe_small = apoe[["RID", "PTID", "VISCODE", "GENOTYPE"]].copy()
print(adas_small.head())
print(dx_small.head())
print(apoe_small.head())

In [ ]:
merged = adas_small.merge(dx_small, on=["RID", "PTID", "VISCODE"], how="inner")
print("Merged shape:", merged.shape)
print(merged.head())

In [ ]:
merged_apoe = merged.merge(apoe_small, on=["RID", "PTID", "VISCODE"], how="left")
print("Merged with APOE shape:", merged_apoe.shape)
print(merged_apoe[["RID", "PTID", "VISCODE", "TOTSCORE", "TOTAL13", "DIAGNOSIS", "GENOTYPE"]].head())
print("\nMissing GENOTYPE:", merged_apoe["GENOTYPE"].isna().sum())

In [ ]:
# baseline and m06 tables from merged_apoe
bl = merged_apoe[merged_apoe["VISCODE"] == "bl"].copy()
m06 = merged_apoe[merged_apoe["VISCODE"] == "m06"].copy()
progression = bl.merge(
    m06[["RID", "PTID", "TOTSCORE", "TOTAL13"]],
    on=["RID", "PTID"],
    how="inner",
    suffixes=("_bl", "_m06")
)
# keep only baseline MCI
progression = progression[progression["DIAGNOSIS"] == 2].copy()
# delta features
progression["delta_ADAS"] = progression["TOTSCORE_m06"] - progression["TOTSCORE_bl"]
progression["delta_ADAS13"] = progression["TOTAL13_m06"] - progression["TOTAL13_bl"]
# label: 1 if patient ever becomes dementia, else 0
labels = {}
for rid in progression["RID"].unique():
    patient_data = merged_apoe[merged_apoe["RID"] == rid]
    labels[rid] = 1 if (patient_data["DIAGNOSIS"] == 3).any() else 0
progression["label"] = progression["RID"].map(labels)

print(progression.head())
print(progression["label"].value_counts())

In [ ]:
print(apoe_small.head(10))
print(apoe_small["VISCODE"].value_counts(dropna=False).head(20))
print(apoe_small["GENOTYPE"].isna().sum(), "missing out of", len(apoe_small))

In [ ]:
apoe_patient = (
    apoe_small[["RID", "PTID", "GENOTYPE"]]
    .dropna(subset=["GENOTYPE"])
    .drop_duplicates(subset=["RID", "PTID"])
    .copy()
)

print(apoe_patient.head())
print("Unique APOE patients:", apoe_patient.shape[0])

In [ ]:
progression = progression.merge(
    apoe_patient,
    on=["RID", "PTID"],
    how="left"
)

print(progression[["RID", "PTID", "GENOTYPE"]].head(10))
print("Missing GENOTYPE in progression:", progression["GENOTYPE"].isna().sum())

In [ ]:
# Create APOE4 carrier feature
progression["APOE4_carrier"] = (
    progression["GENOTYPE"]
    .astype(str)
    .str.contains("4", na=False)
    .astype(int)
)

print(progression[["GENOTYPE", "APOE4_carrier"]].head(10))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

model_data = progression[[
    "TOTSCORE_bl",
    "TOTAL13_bl",
    "delta_ADAS",  #remove from final model
    "delta_ADAS13", #remove from final model
    "APOE4_carrier",
    "label"
]].dropna()

X = model_data[
    "TOTSCORE_bl",
    "TOTAL13_bl",
    "delta_ADAS",
    "delta_ADAS13",
    "APOE4_carrier"
]]
y = model_data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scale_pos_weight = len(y[y == 0]) / len(y[y == 1])

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
print("ADSL SUBJID preview:")
print(adsl[["SUBJID", "USUBJID"]].head(10))

print("\nProgression PTID/RID preview:")
print(progression[["RID", "PTID"]].head(10))

In [ ]:
progression_ids = progression[["RID", "PTID"]].drop_duplicates().copy()
progression_ids["SUBJID_from_PTID"] = progression_ids["PTID"].str.extract(r"_(\d+)$").astype(int)

print(progression_ids.head(10))

In [ ]:
adsl_ids = adsl[["SUBJID"]].drop_duplicates().copy()

check = progression_ids.merge(
    adsl_ids,
    left_on="SUBJID_from_PTID",
    right_on="SUBJID",
    how="inner"
)

print("Matched IDs:", check.shape[0])
print(check.head(10))

In [ ]:
# Keep only needed ADSL columns
adsl_small = adsl[[
    "SUBJID", "AGE", "SEX", "EDUC", "MMSCORE", "MOCA", "CDRSB", "FAQTOTAL"
]].copy()

# Build mapping table from progression PTID -> numeric SUBJID
progression_ids = progression[["RID", "PTID"]].drop_duplicates().copy()
progression_ids["SUBJID"] = progression_ids["PTID"].str.extract(r"_(\d+)$").astype(int)

# Only add SUBJID if it is not already present
if "SUBJID" not in progression.columns:
    progression = progression.merge(
        progression_ids[["RID", "PTID", "SUBJID"]],
        on=["RID", "PTID"],
        how="left"
    )

# Remove duplicate rows in ADSL by SUBJID if any
adsl_small = adsl_small.drop_duplicates(subset=["SUBJID"])

# Merge ADSL features
progression = progression.merge(
    adsl_small,
    on="SUBJID",
    how="left"
)

print(progression[[
    "RID", "PTID", "SUBJID", "AGE", "SEX", "EDUC", "MMSCORE", "MOCA", "CDRSB", "FAQTOTAL"
]].head(10))

print("\nMissing values after ADSL merge:")
print(progression[["AGE", "SEX", "EDUC", "MMSCORE", "MOCA", "CDRSB", "FAQTOTAL"]].isna().sum())

In [ ]:
# Encode sex
progression["SEX"] = progression["SEX"].map({"Male": 1, "Female": 0})

# Fill missing numeric values with median
for col in ["AGE", "EDUC", "MMSCORE", "MOCA", "CDRSB", "FAQTOTAL"]:
    progression[col] = progression[col].fillna(progression[col].median())

In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
# from xgboost import XGBClassifier

# model_data = progression[[
#     "AGE",
#     "SEX",
#     "EDUC",
#     "MMSCORE",
#     "MOCA",
#     "CDRSB",
#     "FAQTOTAL",
#     "TOTSCORE_bl",
#     "TOTAL13_bl",
#     "delta_ADAS",
#     "delta_ADAS13",
#     "APOE4_carrier",
#     "label"
# ]].dropna()

# X = model_data[[
#     "AGE",
#     "SEX",
#     "EDUC",
#     "MMSCORE",
#     "MOCA",
#     "CDRSB",
#     "FAQTOTAL",
#     "TOTSCORE_bl",
#     "TOTAL13_bl",
#     "delta_ADAS",
#     "delta_ADAS13",
#     "APOE4_carrier"
# ]]
# y = model_data["label"]

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# scale_pos_weight = len(y[y == 0]) / len(y[y == 1])

# model = XGBClassifier(
#     n_estimators=300,
#     max_depth=5,
#     learning_rate=0.05,
#     scale_pos_weight=scale_pos_weight,
#     eval_metric="logloss",
#     random_state=42
# )

# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)
# y_prob = model.predict_proba(X_test)[:, 1]

# print("Accuracy:", accuracy_score(y_test, y_pred))
# print("AUC:", roc_auc_score(y_test, y_prob))
# print("\nClassification Report:\n", classification_report(y_test, y_pred))
# print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
print("Progression shape:", progression.shape)

check_cols = [
    "AGE", "SEX", "EDUC", "MMSCORE", "MOCA", "CDRSB", "FAQTOTAL",
    "TOTSCORE_bl", "TOTAL13_bl", "delta_ADAS", "delta_ADAS13",
    "APOE4_carrier", "label"
]

print("\nMissing values by column:")
print(progression[check_cols].isna().sum())

print("\nNon-missing counts by column:")
print(progression[check_cols].notna().sum())

print("\nSEX unique values before mapping check:")
print(progression["SEX"].unique())

print("\nPreview:")
print(progression[check_cols].head())

In [ ]:
print("Progression shape:", progression.shape)

check_cols = [
    "AGE", "SEX", "EDUC", "MMSCORE", "MOCA", "CDRSB", "FAQTOTAL",
    "TOTSCORE_bl", "TOTAL13_bl", "delta_ADAS", "delta_ADAS13",
    "APOE4_carrier", "label"
]

print("\nMissing values by column:")
print(progression[check_cols].isna().sum())

print("\nNon-missing counts by column:")
print(progression[check_cols].notna().sum())

print("\nSEX unique values before mapping check:")
print(progression["SEX"].unique())

print("\nPreview:")
print(progression[check_cols].head())

In [ ]:
model_cols = [
    "AGE",
    "EDUC",
    "MMSCORE",
    "CDRSB",
    "FAQTOTAL",
    "TOTSCORE_bl",
    "TOTAL13_bl",
    "APOE4_carrier",
    "label"
]

model_data = progression[model_cols].dropna()

print("model_data shape:", model_data.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# Initialize model
log_model = LogisticRegression(max_iter=1000, class_weight='balanced')

# Train
log_model.fit(X_train, y_train)

# Predict
y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

# Evaluate
print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("AUC:", roc_auc_score(y_test, y_prob_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_log))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

X = model_data.drop("label", axis=1)
y = model_data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scale_pos_weight = len(y[y == 0]) / len(y[y == 1])

model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("XGBoost Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Initialize model
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=5,
    class_weight='balanced',
    random_state=42
)

# Train
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
print("Random Forest Results")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("AUC:", roc_auc_score(y_test, y_prob_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# --- Get probabilities ---
# XGBoost
y_prob_xgb = model.predict_proba(X_test)[:, 1]

# Logistic Regression
y_prob_log = log_model.predict_proba(X_test)[:, 1]

# Random Forest
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# --- Compute ROC ---
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)
fpr_log, tpr_log, _ = roc_curve(y_test, y_prob_log)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

# --- Compute AUC ---
auc_xgb = auc(fpr_xgb, tpr_xgb)
auc_log = auc(fpr_log, tpr_log)
auc_rf = auc(fpr_rf, tpr_rf)

# --- Plot ---
plt.figure(figsize=(8,6))

plt.plot(fpr_xgb, tpr_xgb, label=f'XGBoost (AUC = {auc_xgb:.3f})')
plt.plot(fpr_log, tpr_log, label=f'Logistic Regression (AUC = {auc_log:.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})')

# Diagonal line (random model)
plt.plot([0,1], [0,1], 'k--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid()

plt.show()

1) Feature importance graph

Correlation matrix


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

feature_cols = [
    "AGE",
    "EDUC",
    "MMSCORE",
    "CDRSB",
    "FAQTOTAL",
    "TOTSCORE_bl",
    "TOTAL13_bl",
    "APOE4_carrier"
]

corr = model_data[feature_cols].corr()

print(corr)

plt.figure(figsize=(8, 6))
plt.imshow(corr, interpolation="nearest")
plt.colorbar()
plt.xticks(range(len(feature_cols)), feature_cols, rotation=45, ha="right")
plt.yticks(range(len(feature_cols)), feature_cols)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary beeswarm
shap.summary_plot(shap_values, X_test)

# Bar summary
shap.summary_plot(shap_values, X_test, plot_type="bar")

In [ ]:
import scipy.stats as stats

group0 = progression[progression["label"] == 0]
group1 = progression[progression["label"] == 1]

features = [
    "AGE", "EDUC", "MMSCORE", "CDRSB", "FAQTOTAL",
    "TOTSCORE_bl", "TOTAL13_bl", "delta_ADAS", "delta_ADAS13"
    
]

for col in features:
    stat, p = stats.ttest_ind(group0[col], group1[col], nan_policy='omit')
    print(f"{col}: p-value = {p:.5f}")

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

table = pd.crosstab(model_data["APOE4_carrier"], model_data["label"])

chi2, p, _, _ = chi2_contingency(table)

print("Chi-square p:", p)

In [ ]:
import pandas as pd
from scipy.stats import chi2_contingency

table = pd.crosstab(progression["APOE4_carrier"], progression["label"])
chi2, p, _, _ = chi2_contingency(table)

print(table)
print("p-value:", p)